# 使用推理模型管理函数调用
OpenAI 现在支持使用[推理模型](https://platform.openai.com/docs/guides/reasoning?api-mode=responses)进行函数调用。推理模型经过训练，能够遵循逻辑思维链，因此更适合复杂或多步骤的任务。
> _o3 和 o4-mini 等推理模型是通过强化学习训练来执行推理的 LLM。推理模型会先思考再回答，在回复用户之前产生一段很长的内部思维链。推理模型擅长复杂问题求解、编程、科学推理以及智能体工作流中的多步骤规划。它们也是我们的轻量级编程智能体 Codex CLI 的最佳模型。_

在大多数情况下，通过 API 使用这些模型非常简单，与使用熟悉的“聊天”模型差不多。 

不过，有一些细微差别需要注意，尤其是在使用函数调用等功能时。 

本 notebook 中的所有示例都使用较新的 [Responses API](https://community.openai.com/t/introducing-the-responses-api/1140929)，它为管理对话状态提供了便捷的抽象。不过，这里介绍的原则在使用较早的 chat completions API 时同样适用。

## 调用推理模型的 API

In [1]:
# pip install openai
# Import libraries 
import json
from openai import OpenAI
from uuid import uuid4
from typing import Callable

client = OpenAI()
MODEL_DEFAULTS = {
    "model": "o4-mini", # 200,000 token context window
    "reasoning": {"effort": "low", "summary": "auto"}, # Automatically summarise the reasoning process. Can also choose "detailed" or "none"
}

我们使用 Responses API 对推理模型进行一次简单调用。
我们指定较低的推理强度（reasoning effort），并通过便捷的 `output_text` 属性获取响应。
我们可以提出后续问题，并使用 `previous_response_id` 让 OpenAI 自动管理对话历史

In [2]:
response = client.responses.create(
    input="Which of the last four Olympic host cities has the highest average temperature?",
    **MODEL_DEFAULTS
)
print(response.output_text)

response = client.responses.create(
    input="what about the lowest?",
    previous_response_id=response.id,
    **MODEL_DEFAULTS
)
print(response.output_text)

Among the last four Summer Olympic host cities—Tokyo (2020), Rio de Janeiro (2016), London (2012) and Beijing (2008)—Rio de Janeiro has by far the warmest climate. Average annual temperatures are roughly:

• Rio de Janeiro: ≈ 23 °C  
• Tokyo: ≈ 16 °C  
• Beijing: ≈ 13 °C  
• London: ≈ 11 °C  

So Rio de Janeiro has the highest average temperature.
Among those four, London has the lowest average annual temperature, at about 11 °C.


简单又方便！

我们提出的问题相对复杂，可能需要模型推理出一个计划并逐步执行，但这些推理对我们是隐藏的——我们只是多等一会儿才看到响应。 

不过，如果检查输出，我们会发现模型使用了一组隐藏的“推理” token，这些 token 包含在模型的上下文窗口中，但不会向作为最终用户的我们公开。
我们可以在响应中看到这些 token 的数量以及推理摘要（但看不到实际使用的 token 内容）。

In [3]:
print(next(rx for rx in response.output if rx.type == 'reasoning').summary[0].text)
response.usage.to_dict()

**Determining lowest temperatures**

The user is asking about the lowest average temperatures of the last four Olympic host cities: Tokyo, Rio, London, and Beijing. I see London has the lowest average temperature at around 11°C. If I double-check the annual averages: Rio is about 23°C, Tokyo is around 16°C, and Beijing is approximately 13°C. So, my final answer is London with an average of roughly 11°C. I could provide those approximate values clearly for the user.


{'input_tokens': 136,
 'input_tokens_details': {'cached_tokens': 0},
 'output_tokens': 89,
 'output_tokens_details': {'reasoning_tokens': 64},
 'total_tokens': 225}

了解这些推理 token 很重要，因为这意味着与传统聊天模型相比，我们会更快地消耗可用的上下文窗口。

## 调用自定义函数
如果我们向模型提出一个复杂请求，而这个请求还需要使用自定义工具，会发生什么？
* 假设我们还有更多关于奥运城市的问题，同时我们有一个内部数据库，其中包含每个城市的 ID。
* 模型可能需要在推理过程中途调用我们的工具，然后才能返回结果。
* 我们来编写一个生成随机 UUID 的函数，并让模型对这些 UUID 进行推理。 


In [4]:

def get_city_uuid(city: str) -> str:
    """Just a fake tool to return a fake UUID"""
    uuid = str(uuid4())
    return f"{city} ID: {uuid}"

# The tool schema that we will pass to the model
tools = [
    {
        "type": "function",
        "name": "get_city_uuid",
        "description": "Retrieve the internal ID for a city from the internal database. Only invoke this function if the user needs to know the internal ID for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "The name of the city to get information about"}
            },
            "required": ["city"]
        }
    }
]

# This is a general practice - we need a mapping of the tool names we tell the model about, and the functions that implement them.
tool_mapping = {
    "get_city_uuid": get_city_uuid
}

# Let's add this to our defaults so we don't have to pass it every time
MODEL_DEFAULTS["tools"] = tools

response = client.responses.create(
    input="What's the internal ID for the lowest-temperature city?",
    previous_response_id=response.id,
    **MODEL_DEFAULTS)
print(response.output_text)


这次我们没有得到 `output_text`。来看看响应的输出

In [5]:
response.output

[ResponseReasoningItem(id='rs_68246219e8288191af051173b1d53b3f0c4fbdb0d4a46f3c', summary=[], type='reasoning', status=None),
 ResponseFunctionToolCall(arguments='{"city":"London"}', call_id='call_Mx6pyTjCkSkmASETsVASogoC', name='get_city_uuid', type='function_call', id='fc_6824621b8f6c8191a8095df7230b611e0c4fbdb0d4a46f3c', status='completed')]

除了推理步骤之外，模型还成功识别出需要调用工具，并返回了要发送给函数调用的指令。 

我们来调用该函数，并将结果发送给模型，让它继续推理。
函数响应是一种特殊的消息，因此我们需要将下一条消息构造成一种特殊的输入：
```json
{
    "type": "function_call_output",
    "call_id": function_call.call_id,
    "output": tool_output
}
```

In [6]:
# Extract the function call(s) from the response
new_conversation_items = []
function_calls = [rx for rx in response.output if rx.type == 'function_call']
for function_call in function_calls:
    target_tool = tool_mapping.get(function_call.name)
    if not target_tool:
        raise ValueError(f"No tool found for function call: {function_call.name}")
    arguments = json.loads(function_call.arguments) # Load the arguments as a dictionary
    tool_output = target_tool(**arguments) # Invoke the tool with the arguments
    new_conversation_items.append({
        "type": "function_call_output",
        "call_id": function_call.call_id, # We map the response back to the original function call
        "output": tool_output
    })

In [7]:
response = client.responses.create(
    input=new_conversation_items,
    previous_response_id=response.id,
    **MODEL_DEFAULTS
)
print(response.output_text)


The internal ID for London is 816bed76-b956-46c4-94ec-51d30b022725.


这在这里效果很好——因为我们知道模型只需一次函数调用就能作出响应——但我们还需要考虑到推理完成可能需要执行多次工具调用的情况。

我们再添加一个运行网页搜索的调用。

OpenAI 的网页搜索工具并不能直接用于推理模型（截至 2025 年 5 月——这可能很快会改变），但使用 4o mini 或其他支持网页搜索的模型来创建自定义网页搜索函数并不难。

In [8]:
def web_search(query: str) -> str:
    """Search the web for information and return back a summary of the results"""
    result = client.responses.create(
        model="gpt-4o-mini",
        input=f"Search the web for '{query}' and reply with only the result.",
        tools=[{"type": "web_search_preview"}],
    )
    return result.output_text

tools.append({
        "type": "function",
        "name": "web_search",
        "description": "Search the web for information and return back a summary of the results",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The query to search the web for."}
            },
            "required": ["query"]
        }
    })
tool_mapping["web_search"] = web_search


## 串行执行多个函数

部分 OpenAI 模型支持 `parallel_tool_calls` 参数，允许模型返回一个函数数组，我们随后可以并行执行这些函数。然而，推理模型可能会产生一系列必须串行执行的函数调用，尤其是某些步骤可能依赖于前面步骤的结果。
因此，我们应当定义一种通用模式，用于处理任意复杂的推理工作流：
* 在对话的每一步，启动一个循环
* 如果响应中包含函数调用，我们必须假定推理仍在进行，应将函数结果（以及任何中间推理）反馈给模型以进行进一步推理
* 如果没有函数调用，而是收到了类型为 'message' 的 response.output，我们就可以放心地认为智能体已完成推理，并跳出循环

In [9]:
# Let's wrap our logic above into a function which we can use to invoke tool calls.
def invoke_functions_from_response(response,
                                   tool_mapping: dict[str, Callable] = tool_mapping
                                   ) -> list[dict]:
    """Extract all function calls from the response, look up the corresponding tool function(s) and execute them.
    (This would be a good place to handle asynchroneous tool calls, or ones that take a while to execute.)
    This returns a list of messages to be added to the conversation history.
    """
    intermediate_messages = []
    for response_item in response.output:
        if response_item.type == 'function_call':
            target_tool = tool_mapping.get(response_item.name)
            if target_tool:
                try:
                    arguments = json.loads(response_item.arguments)
                    print(f"Invoking tool: {response_item.name}({arguments})")
                    tool_output = target_tool(**arguments)
                except Exception as e:
                    msg = f"Error executing function call: {response_item.name}: {e}"
                    tool_output = msg
                    print(msg)
            else:
                msg = f"ERROR - No tool registered for function call: {response_item.name}"
                tool_output = msg
                print(msg)
            intermediate_messages.append({
                "type": "function_call_output",
                "call_id": response_item.call_id,
                "output": tool_output
            })
        elif response_item.type == 'reasoning':
            print(f'Reasoning step: {response_item.summary}')
    return intermediate_messages

现在来演示前面讨论的循环概念。

In [10]:
initial_question = (
    "What are the internal IDs for the cities that have hosted the Olympics in the last 20 years, "
    "and which of those cities have recent news stories (in 2025) about the Olympics? "
    "Use your internal tools to look up the IDs and the web search tool to find the news stories."
)

# We fetch a response and then kick off a loop to handle the response
response = client.responses.create(
    input=initial_question,
    **MODEL_DEFAULTS,
)
while True:   
    function_responses = invoke_functions_from_response(response)
    if len(function_responses) == 0: # We're done reasoning
        print(response.output_text)
        break
    else:
        print("More reasoning required, continuing...")
        response = client.responses.create(
            input=function_responses,
            previous_response_id=response.id,
            **MODEL_DEFAULTS
        )

Reasoning step: []
Invoking tool: get_city_uuid({'city': 'Beijing'})
More reasoning required, continuing...
Reasoning step: []
Invoking tool: get_city_uuid({'city': 'London'})
More reasoning required, continuing...
Reasoning step: []
Invoking tool: get_city_uuid({'city': 'Rio de Janeiro'})
More reasoning required, continuing...
Reasoning step: []
Invoking tool: get_city_uuid({'city': 'Tokyo'})
More reasoning required, continuing...
Reasoning step: []
Invoking tool: get_city_uuid({'city': 'Paris'})
More reasoning required, continuing...
Reasoning step: []
Invoking tool: get_city_uuid({'city': 'Turin'})
More reasoning required, continuing...
Reasoning step: []
Invoking tool: get_city_uuid({'city': 'Vancouver'})
More reasoning required, continuing...
Reasoning step: []
Invoking tool: get_city_uuid({'city': 'Sochi'})
More reasoning required, continuing...
Reasoning step: []
Invoking tool: get_city_uuid({'city': 'Pyeongchang'})
More reasoning required, continuing...
Reasoning step: []
Invok

## 手动编排对话
到目前为止一切顺利！看着模型暂停执行去运行一个函数然后再继续，真的很酷。 
实际上，上面的示例相当简单，而生产环境中的用例可能要复杂得多：
* 上下文窗口可能会变得过大，我们可能希望裁剪较旧且相关性较低的消息，或对目前为止的对话进行摘要
* 我们可能希望允许用户在对话中来回浏览并重新生成回答
* 出于审计目的，我们可能希望将消息存储在自己的数据库中，而不是依赖 OpenAI 的存储和编排
* 等等

在这些情况下，我们可能希望完全掌控对话。我们可以不使用 `previous_message_id`，而是将 API 视为“无状态”的，自行创建并维护一个对话条目数组，每次都将其作为输入发送给模型。

这就带来了一些推理模型特有的细微差别需要考虑。 
* 尤其重要的是，我们必须在对话历史中保留所有推理和函数调用响应。
* 模型正是借此跟踪它已经执行过哪些思维链步骤。如果没有包含这些内容，API 会报错。

我们再运行一遍上面的示例，这次自己编排消息并跟踪 token 用量。

---
*注意，下面的代码以可读性为目标进行组织——在实践中，你可能需要考虑更完善的工作流来处理边界情况*

In [11]:
# Let's initialise our conversation with the first user message
total_tokens_used = 0
user_messages = [
    (
        "Of those cities that have hosted the summer Olympic games in the last 20 years - "
        "do any of them have IDs beginning with a number and a temperate climate? "
        "Use your available tools to look up the IDs for each city and make sure to search the web to find out about the climate."
    ),
    "Great thanks! We've just updated the IDs - could you please check again?"
    ]

conversation = []
for message in user_messages:
    conversation_item = {
        "role": "user",
        "type": "message",
        "content": message
    }
    print(f"{'*' * 79}\nUser message: {message}\n{'*' * 79}")
    conversation.append(conversation_item)
    while True: # Response loop
        response = client.responses.create(
            input=conversation,
            **MODEL_DEFAULTS
        )
        total_tokens_used += response.usage.total_tokens
        reasoning = [rx.to_dict() for rx in response.output if rx.type == 'reasoning']
        function_calls = [rx.to_dict() for rx in response.output if rx.type == 'function_call']
        messages = [rx.to_dict() for rx in response.output if rx.type == 'message']
        if len(reasoning) > 0:
            print("More reasoning required, continuing...")
            # Ensure we capture any reasoning steps
            conversation.extend(reasoning)
            print('\n'.join(s['text'] for r in reasoning for s in r['summary']))
        if len(function_calls) > 0:
            function_outputs = invoke_functions_from_response(response)
            # Preserve order of function calls and outputs in case of multiple function calls (currently not supported by reasoning models, but worth considering)
            interleaved = [val for pair in zip(function_calls, function_outputs) for val in pair]
            conversation.extend(interleaved)
        if len(messages) > 0:
            print(response.output_text)
            conversation.extend(messages)
        if len(function_calls) == 0:  # No more functions = We're done reasoning and we're ready for the next user message
            break
print(f"Total tokens used: {total_tokens_used} ({total_tokens_used / 200_000:.2%} of o4-mini's context window)")

*******************************************************************************
User message: Of those cities that have hosted the summer Olympic games in the last 20 years - do any of them have IDs beginning with a number and a temperate climate? Use your available tools to look up the IDs for each city and make sure to search the web to find out about the climate.
*******************************************************************************
More reasoning required, continuing...
**Clarifying Olympic Cities**

The user is asking about cities that hosted the Summer Olympics in the last 20 years. The relevant years to consider are 2004 Athens, 2008 Beijing, 2012 London, 2016 Rio de Janeiro, and 2020 Tokyo. If we're considering 2025, then 2004 would actually be 21 years ago, so I should focus instead on the years from 2005 onwards. Therefore, the cities to include are Beijing, London, Rio, and Tokyo. I’ll exclude Paris since it hasn’t hosted yet.
Reasoning step: [Summary(text="**Clarif

## 总结
在本 cookbook 中，我们介绍了如何将函数调用与 OpenAI 的推理模型结合起来，演示依赖外部数据源（包括网页搜索）的多步骤任务。

重要的是，我们讲解了函数调用过程中推理模型特有的细微差别，具体来说：
* 模型可能会选择串行执行多次函数调用或推理步骤，而某些步骤可能依赖于前面步骤的结果
* 我们无法预知会有多少个这样的步骤，因此必须用循环来处理响应
* Responses API 通过 `previous_response_id` 参数让编排变得简单，但在需要手动控制时，务必保持对话条目的正确顺序，以保留“思维链”

---

这里使用的示例相当简单，但你可以想象如何将这一技术扩展到更多真实世界的用例，例如：

* 查询客户的交易历史和近期往来记录，判断其是否有资格获得促销优惠
* 调取近期交易日志、地理位置数据和设备元数据，评估一笔交易存在欺诈的可能性
* 查阅内部 HR 数据库，获取员工的福利使用情况、工龄和近期政策变更，以回答个性化的 HR 问题
* 阅读内部仪表盘、竞争对手新闻源和市场分析，为高管编写契合其关注领域的每日简报